# Day 2 — Predict, evaluate and compare

All data are synthetic. Run from top to bottom. Work in pairs and pause after each result to explain its meaning. Use TEACHING_GUIDE.md and TASK_CARDS.md for timing. Optional sections are marked. Numerical outputs are examples, not evidence about real operations.

## 1. Define a prediction problem

Predict route duration before departure. Inputs are planned distance and stops; actual duration and the derived delayed flag are outcomes. Independent synthetic routes allow random splitting here. Real time-dependent routes would require chronological evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, confusion_matrix, classification_report, accuracy_score
rng=np.random.default_rng(22)
X=pd.DataFrame({'distance_km':rng.uniform(1,40,400),'stops':rng.integers(1,9,400)})
y=12+2.4*X.distance_km+4*X.stops+rng.normal(0,9,len(X))
delayed=(y>85).astype(int)
X.loc[::23,'distance_km']=np.nan
train_ids,test_ids=train_test_split(np.arange(len(X)),test_size=.25,random_state=42,stratify=delayed)
X_train,X_test=X.iloc[train_ids],X.iloc[test_ids]
y_train,y_test=y.iloc[train_ids],y.iloc[test_ids]
c_train,c_test=delayed.iloc[train_ids],delayed.iloc[test_ids]
print('Rows:',len(X),'Training:',len(X_train),'Final test:',len(X_test))
print(X.head().to_string(index=False))

## 2. Regression and a baseline

The imputer learns only from training rows. Cross-validation measures performance within training data. Larger scikit-learn scores are better, so neg_mean_absolute_error is negated for readable MAE. This lab preselects linear regression; the final test is used once.

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
reg=make_pipeline(SimpleImputer(strategy='median'),LinearRegression())
cv_mae=-cross_val_score(reg,X_train,y_train,cv=cv,scoring='neg_mean_absolute_error')
reg.fit(X_train,y_train)
base=DummyRegressor(strategy='mean').fit(X_train,y_train)
print('Validation MAE by fold:',np.round(cv_mae,2))
print('Baseline final-test MAE:',round(mean_absolute_error(y_test,base.predict(X_test)),2))
print('Regression final-test MAE:',round(mean_absolute_error(y_test,reg.predict(X_test)),2))
plt.figure(figsize=(6,4));plt.scatter(y_test,reg.predict(X_test),alpha=.6)
plt.xlabel('Actual minutes');plt.ylabel('Predicted minutes');plt.title('Held-out route durations');plt.tight_layout();plt.show()

## 3. Classification and model selection

Compare logistic regression, a tree and a forest on training folds. Tune a small forest grid on those same training data. Select a candidate using validation only. Final test metrics do not feed further choices. Accuracy can hide different types of error.

In [ ]:
scv=StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
candidates={
 'logistic':make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(max_iter=1000)),
 'tree':make_pipeline(SimpleImputer(),DecisionTreeClassifier(max_depth=3,random_state=42)),
 'forest':make_pipeline(SimpleImputer(),RandomForestClassifier(n_estimators=60,max_depth=5,random_state=42,n_jobs=1))}
scores={name:cross_val_score(model,X_train,c_train,cv=scv,scoring='f1').mean() for name,model in candidates.items()}
search=GridSearchCV(candidates['forest'],{'randomforestclassifier__max_depth':[3,5,None],'randomforestclassifier__min_samples_leaf':[2,5]},cv=scv,scoring='f1',n_jobs=1)
search.fit(X_train,c_train)
scores['tuned_forest']=search.best_score_
candidates['tuned_forest']=search.best_estimator_
selected_name=max(scores,key=scores.get)
selected=candidates[selected_name].fit(X_train,c_train)
pred=selected.predict(X_test)
dummy=DummyClassifier(strategy='most_frequent').fit(X_train,c_train)
print('Validation F1:',{k:round(v,3) for k,v in scores.items()})
print('Selected before testing:',selected_name)
print('Baseline accuracy:',round(accuracy_score(c_test,dummy.predict(X_test)),3))
print('Confusion matrix: rows=actual, columns=predicted; class order [0,1]')
print(confusion_matrix(c_test,pred,labels=[0,1]))
print(classification_report(c_test,pred,zero_division=0))

## 4. Clustering and PCA — guided extension

Unsupervised analysis uses only training features here. Standardisation prevents kilometres dominating stop counts solely because of units. K=3 is chosen for demonstration, not discovered as a true number of customer types. PCA rotates/compresses variation; it does not establish business meaning.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
prep=make_pipeline(SimpleImputer(),StandardScaler())
Z=prep.fit_transform(X_train)
clusters=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(Z)
pca=PCA(n_components=1).fit(Z)
print('Cluster counts:',np.bincount(clusters))
print('Variance retained by one component:',round(pca.explained_variance_ratio_[0],3))
plt.figure(figsize=(6,4));plt.scatter(Z[:,0],Z[:,1],c=clusters,cmap='viridis',s=18)
plt.xlabel('Standardised distance');plt.ylabel('Standardised stops');plt.title('Exploratory route groups');plt.tight_layout();plt.show()

# Your pair challenge

Use the working examples above. Complete the tasks below in new cells. For model experiments use training/validation data; do not optimise against a final test set.

A. Explain the two regression MAEs in minutes. Why is the baseline necessary?

In [ ]:
# Add your experiment or calculation here.

B. From the confusion matrix calculate false negatives: actual delayed, predicted not delayed. Explain their operational cost. Do not alter the threshold using this test set.

In [ ]:
# Add your experiment or calculation here.

C. On training folds only, compare tree depths 1, 3 and 8. Record training accuracy and mean validation F1. A larger gap suggests overfitting; it does not prove that depth 8 always performs worse.

In [ ]:
# Add your experiment or calculation here.

Extension. Compare k=2 and k=4 on Z. Explain why different partitions are not automatically errors.

In [ ]:
# Add your experiment or calculation here.